In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import torch
import torch.nn as nn
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from accelerate import Accelerator
import os
from PIL import Image
from torchvision import transforms
import numpy as np
from tqdm.auto import tqdm
import random
import matplotlib.pyplot as plt
from torchvision.transforms import functional as TF
from safetensors.torch import load_file
from safetensors.torch import save_file

class OutpaintDataset(Dataset):
    def __init__(self, root_dir, img_size=512, masks_per_image=100):
        self.img_files = [os.path.join(root_dir, f) for f in os.listdir(root_dir)]
        self.img_size = img_size
        self.masks_per_image = masks_per_image

        self.transform = transforms.Compose([
            transforms.RandomChoice([
                transforms.Lambda(lambda x: x),
                transforms.Lambda(lambda x: TF.rotate(x, 90)),
                transforms.Lambda(lambda x: TF.rotate(x, 180)),
                transforms.Lambda(lambda x: TF.rotate(x, 270))
            ]),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.img_files) * self.masks_per_image

    def _gen_random_irregular_mask(self, H, W, min_area=0.01, max_area=0.1, num_vertices=10):
      cx = random.uniform(0.2, 0.8) * W
      cy = random.uniform(0.2, 0.8) * H

      target_area_ratio = random.uniform(min_area, max_area)
      target_area = target_area_ratio * H * W

      theta0 = random.uniform(0, 2*np.pi)

      angles = np.sort(np.random.uniform(0, 2*np.pi, num_vertices))

      avg_r = np.sqrt(target_area / (math.pi * 0.3))

      radii = []
      for a in angles:
        diff = abs(a - theta0)
        diff = min(diff, 2*np.pi - diff)
        slim_factor = 3.0
        r = avg_r * (0.7 + 0.6 * np.exp(-slim_factor * diff))
        r *= (0.8 + 0.4 * np.random.rand())
        radii.append(r)


      points = []
      for a, r in zip(angles, radii):
        x = cx + r * np.cos(a)
        y = cy + r * np.sin(a)
        points.append([int(np.clip(x, 0, W-1)), int(np.clip(y, 0, H-1))])

      polygon = np.array(points)
      mask = np.zeros((H, W), dtype=np.uint8)

      for y in range(H):
        xs = []
        for i in range(num_vertices):
            x1, y1 = polygon[i]
            x2, y2 = polygon[(i+1) % num_vertices]
            if y1 == y2:
                continue
            if (y >= min(y1,y2)) and (y < max(y1,y2)):
                x_int = x1 + (y - y1)*(x2 - x1)/(y2 - y1)
                xs.append(int(x_int))
        xs.sort()
        for i in range(0, len(xs), 2):
            if i+1 < len(xs):
                mask[y, xs[i]:xs[i+1]] = 1

      return torch.tensor(mask).float().unsqueeze(0)


    def __getitem__(self, idx):
        img_idx = idx // self.masks_per_image
        img = Image.open(self.img_files[img_idx]).convert("RGB")
        img = self.transform(img)
        C, H, W = img.shape
        mask = self._gen_random_irregular_mask(H, W)
        masked_img = img * (1 - mask)
        return masked_img, img, mask

In [3]:
class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1

    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)

    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, num_patches, dim]

    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=736, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64), # [B, 64, 64, 64]
            nn.AvgPool2d(2), # [B, 64, 32, 32]
            ResidualBlock(64, 128),
            nn.AvgPool2d(2), # [B, 128, 16, 16]
            ResidualBlock(128, 256),
            nn.AvgPool2d(2), # [B, 256, 8, 8]
            nn.Conv2d(256, out_channels, kernel_size=1) # [B, 736, 8, 8]
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),  # [B, 736, 64]
            Transpose(-1, -2),   # [B, 64, 736]
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        feat = self.encoder(x)          # [B, 736, 8, 8]
        tokens = self.proj(feat)        # [B, 64, 736]
        tokens = self.pos_embed(tokens) # [B, 64, 736]
        tokens = self.norm(tokens)
        return tokens

class CoordEncoder(nn.Module):
    def __init__(self, embed_dim=32, num_tokens=64):
        super().__init__()
        self.num_tokens = num_tokens
        self.embed_dim = embed_dim

        self.mlp = nn.Sequential(
            nn.Linear(64 * 64, 2048),
            nn.GELU(),
            nn.Linear(2048, num_tokens * embed_dim)  # 64 * 32
        )

    def forward(self, mask):
        mask_ds = F.interpolate(mask, size=(64, 64), mode="nearest")  # (B,1,64,64)
        B = mask_ds.shape[0]
        x = mask_ds.view(B, -1)  # (B, 4096)
        x = self.mlp(x)  # (B, 64*32)
        x = x.view(B, self.num_tokens, self.embed_dim)  # (B,64,32)
        return x


In [4]:
class OutpaintTrainer_new:
    def __init__(self, pretrained_path="runwayml/stable-diffusion-v1-5"):

        self.accelerator = Accelerator(mixed_precision="fp16")
        self.loss_history = []
        self.val_loss_history = []

        train_data = OutpaintDataset("drive/MyDrive/resize1")
        val_data   = OutpaintDataset("drive/MyDrive/Testset")

        self.train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
        self.val_loader   = DataLoader(val_data, batch_size=4, shuffle=False)

        # --- Models ---
        self.vae = AutoencoderKL.from_pretrained(pretrained_path, subfolder="vae")
        self.unet = UNet2DConditionModel.from_pretrained(pretrained_path, subfolder="unet")

        # latent channels often = 4
        latent_c = self.vae.config.latent_channels

        self.cond_proj = CondEncoder(in_channels=latent_c, out_channels=736)
        self.coord_encoder = CoordEncoder(embed_dim=32, num_tokens=64)

        # move to accelerator
        components = [
            self.vae, self.unet,
            self.cond_proj, self.coord_encoder,
            self.train_loader, self.val_loader
        ]
        (
            self.vae, self.unet,
            self.cond_proj, self.coord_encoder,
            self.train_loader, self.val_loader
        ) = self.accelerator.prepare(*components)

        self.optimizer = torch.optim.AdamW(
            list(self.unet.parameters()) +
            list(self.cond_proj.parameters()) +
            list(self.coord_encoder.parameters()),
            lr=2e-5
        )
        self.optimizer = self.accelerator.prepare(self.optimizer)

        self.noise_scheduler = DDPMScheduler.from_pretrained(pretrained_path, subfolder="scheduler")
        self.vae.requires_grad_(False)

    def load_partial_checkpoint(self, ckpt_dir):

       unet_path = os.path.join(ckpt_dir, "model_1.safetensors")
       cond_path = os.path.join(ckpt_dir, "model_3.safetensors")

       # ---- load UNET ----
       if os.path.exists(unet_path):
          print(f"Loading UNET from {unet_path}")
          state = load_file(unet_path)
          missing, unexpected = self.unet.load_state_dict(state, strict=True)
          print("UNET loaded.")
          print("Missing:", missing)
          print("Unexpected:", unexpected)

       # ---- load CondEncoder ----
       if os.path.exists(cond_path):
          print(f"Loading CondEncoder from {cond_path}")
          state = load_file(cond_path)
          missing, unexpected = self.cond_proj.load_state_dict(state, strict=True)
          print("CondEncoder loaded.")
          print("Missing:", missing)
          print("Unexpected:", unexpected)

       print("Partial checkpoint loaded (UNet + CondEncoder). CoordEncoder ignored.")



    def save_best_checkpoint(self, save_dir):
      os.makedirs(save_dir, exist_ok=True)

      # --- UNET ---
      unet_path = os.path.join(save_dir, "unet.safetensors")
      save_file(self.unet.state_dict(), unet_path)
      print(f"[Saved] UNet → {unet_path}")

      # --- CondEncoder ---
      cond_path = os.path.join(save_dir, "condencoder.safetensors")
      save_file(self.cond_proj.state_dict(), cond_path)
      print(f"[Saved] CondEncoder → {cond_path}")

      # --- CoordEncoder ---
      coord_path = os.path.join(save_dir, "coordencoder.safetensors")
      save_file(self.coord_encoder.state_dict(), coord_path)
      print(f"[Saved] CoordEncoder → {coord_path}")

      print("Saved best checkpoint.")

    # -----------------------------------------
    # Validation step + plot mask
    # -----------------------------------------
    def validate_step(self):
        self.unet.eval()
        tot = 0
        cnt = 0

        with torch.no_grad():
            for bi, batch in enumerate(self.val_loader):
                loss = self.train_step(batch, training=False)
                tot += loss.item()
                cnt += 1

                # plot mask for first batch
                if bi == 0:
                    _, _, mask_1 = batch
                    plt.imshow(mask_1[0,0].cpu(), cmap='gray')
                    plt.title("Validation Mask")
                    plt.axis("off")
                    plt.show()

        return tot / max(cnt, 1)

    # -----------------------------------------
    # Train step
    # -----------------------------------------
    def train_step(self, batch, training=True):
        masked_imgs, target_imgs, mask = batch

        # target latents
        with torch.no_grad():
            target_latents = self.vae.encode(target_imgs).latent_dist.sample()
            target_latents = target_latents * self.vae.config.scaling_factor

        B, C, lh, lw = target_latents.shape

        latent_mask = F.interpolate(mask, size=(lh, lw), mode="nearest")
        latent_mask = latent_mask.expand(-1, target_latents.shape[1], -1, -1)  # (B,4,lh,lw)

        noise = torch.randn_like(target_latents)
        timesteps = torch.randint(
            0, self.noise_scheduler.config.num_train_timesteps,
            (B,), device=target_latents.device
        )

        noisy_latents = self.noise_scheduler.add_noise(
            target_latents * latent_mask,
            noise * latent_mask,
            timesteps
        )
        noisy_latents = target_latents * (1-latent_mask) + noisy_latents

        # masked latents for CondEncoder
        with torch.no_grad():
            masked_latents = self.vae.encode(masked_imgs).latent_dist.sample()
            masked_latents = masked_latents * self.vae.config.scaling_factor

        # Encode conditions
        cond_tokens = self.cond_proj(masked_latents)      # (B,64,736)
        coord_tokens = self.coord_encoder(mask)         # (B,64,32)

        condition = torch.cat([cond_tokens, coord_tokens], dim=-1)  # (B,64,768)

        noise_pred = self.unet(noisy_latents, timesteps, encoder_hidden_states=condition).sample

        loss = F.mse_loss(noise_pred * latent_mask, noise * latent_mask)
        return loss

    # -----------------------------------------
    # Full training loop
    # -----------------------------------------
    def train(self, epochs=20):
        best_val_loss = float("inf")
        for ep in range(epochs):
            self.unet.train()
            pbar = tqdm(self.train_loader, desc=f"Epoch {ep}")

            losses = []
            for batch in pbar:
                with self.accelerator.accumulate(self.unet):
                    loss = self.train_step(batch)
                    self.accelerator.backward(loss)
                    self.optimizer.step()
                    self.optimizer.zero_grad()

                losses.append(loss.item())
                pbar.set_postfix({"loss": np.mean(losses)})

            train_loss = np.mean(losses)
            self.loss_history.append(train_loss)

            val_loss = self.validate_step()
            self.val_loss_history.append(val_loss)

            print(f"Epoch {ep}: train={train_loss:.4f}, val={val_loss:.4f}")

            self.plot_loss()

            # ---------- SAVE BEST ----------
            if val_loss < best_val_loss:
              best_val_loss = val_loss
              print("New best val loss, Saving checkpoint...")
              self.save_best_checkpoint("drive/MyDrive/inpaint_best/")

    def plot_loss(self):
        plt.figure(figsize=(6,5))
        plt.plot(self.loss_history, label="train")
        plt.plot(self.val_loss_history, label="val")
        plt.legend()
        plt.show()


In [7]:
    torch.cuda.empty_cache()
    import gc
    gc.collect()

128

In [ ]:
#=======================================
# Training Execution
#=======================================
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

trainer = OutpaintTrainer_new()
trainer.load_partial_checkpoint("drive/MyDrive/checkpoint_LR")
optimizer = torch.optim.AdamW([
            {"params": trainer.unet.parameters(), "lr": 5e-6},
            {"params": trainer.coord_encoder.parameters(), "lr": 2e-5},
            {"params": trainer.cond_proj.parameters(), "lr": 1e-6},
          ])
trainer.optimizer = trainer.accelerator.prepare(optimizer)

In [ ]:
trainer.train(10)

In [ ]:
trainer.save_best_checkpoint("drive/MyDrive/inpaint_best/")